# Feature — Reward-Hacking Defenses + Groundedness Scoring

Three anti-gaming defenses (score clamping, judge-prompt rotation, relative/batch scoring) plus groundedness scoring, each exercised for real against a local Qwen2.5-0.5B-Instruct model (no external API key needed).


In [1]:
"""
Feature — Reward-hacking defenses + Groundedness scoring.
===========================================================

Three anti-gaming defenses used in AgentTune's reward/judge code, each exercised for
real (local Qwen2.5-0.5B-Instruct where an LLM call is needed, no external API key):

1. Score clamping — `agenttune.rag.rewards.t3_rewards.necessity_reward` /
   `frugality_reward` cap the reward instead of letting it run away when an agent
   over-searches relative to the optimal count.
2. Prompt rotation — `agenttune.eval.agentic.trajectory_eval.TrajectoryEvaluator`
   randomly rotates between 3 differently-worded judge system prompts per call, so a
   policy can't overfit to gaming one fixed prompt.
3. Relative (batch) scoring — `agenttune.agentic.rewards.llm_judge.LLMJudge`'s
   `mode="relative"` shows the judge all trajectories in a batch at once instead of
   scoring each in isolation.

Plus groundedness scoring — `agenttune.rag.rewards.judge_eval`'s rubric checks
whether an answer's claims are actually supported by retrieved passages, rather than
trusting the model's own confidence. The library's `build_groq_judge` needs a
GROQ_API_KEY (not available here); this script builds an equivalent local
`LLMJudge` using the same `GROUNDEDNESS_RUBRIC` against a real local model instead.
"""
import asyncio
import nest_asyncio
nest_asyncio.apply()

In [2]:
from agenttune.rag.rewards.t3_rewards import necessity_reward, frugality_reward
from agenttune.rag.rewards.judge_eval import GROUNDEDNESS_RUBRIC
from agenttune.agentic.rewards.llm_judge import LLMJudge
from agenttune.eval.agentic.trajectory_eval import TrajectoryEvaluator
from agenttune.agentic.inference.transformers_engine import TransformersEngine

In [3]:
from _real_backends import real_rollout_engine

In [4]:
def main():
    # ── 1) Score clamping (t3_rewards) ──────────────────────────────────────
    print("── 1) Score clamping: necessity/frugality rewards ──")
    # Same correct answer, wildly different search counts.
    completions = ["<answer>Paris</answer>"] * 3
    tool_call_counts = [1, 3, 12]  # optimal, a bit much, way over-retrieval
    gold = ["Paris"] * 3
    requires_search = [True, True, True]

    necessity = necessity_reward(None, completions, tool_call_counts=tool_call_counts,
                                  gold_answer=gold, requires_search=requires_search)
    frugality = frugality_reward(None, completions, tool_call_counts=tool_call_counts,
                                  gold_answer=gold, requires_search=requires_search)
    for n, f, c in zip(necessity, frugality, tool_call_counts):
        print(f"  searches={c:2d}  necessity_reward={n:.3f}  frugality_reward={f:.3f}")
    print("  -> reward does not grow (or drops) past the optimal search count: over-retrieval is capped, not free.\n")

    # ── 2) Prompt rotation (TrajectoryEvaluator) ────────────────────────────
    print("── 2) Prompt rotation: TrajectoryEvaluator's judge prompt ──")
    engine = TransformersEngine("Qwen/Qwen2.5-0.5B-Instruct")
    evaluator = TrajectoryEvaluator(engine=engine)

    trajectory = {
        "trajectory_id": "t1",
        "task": "What is the capital of France?",
        "steps": [
            {"action": {"name": "search", "arguments": {"query": "capital of France"}},
             "observation": "Paris is the capital of France."},
            {"action": {"name": "finish", "arguments": {"answer": "Paris"}}},
        ],
    }

    async def run_rotation_demo():
        results = []
        for _ in range(4):
            r = await evaluator._evaluate_single(trajectory)
            results.append(r.rotation_id)
        return results

    rotation_ids = asyncio.run(run_rotation_demo())
    print(f"  rotation_id across 4 real judge calls: {rotation_ids}")
    print(f"  -> {'rotates across calls' if len(set(rotation_ids)) > 1 else 'only one rotation id seen this run (still random each call)'}, "
          f"picked from 3 differently-worded judge prompts.\n")

    # ── 3) Relative (batch) scoring (LLMJudge) ──────────────────────────────
    print("── 3) Relative scoring: LLMJudge mode='relative' vs 'absolute' ──")
    rollout_engine = real_rollout_engine()
    judge = LLMJudge(rollout_engine=rollout_engine)

    task = "What is the capital of France?"
    good_traj = ["search(capital of France) -> 'Paris is the capital of France.'",
                 "answer: Paris"]
    bad_traj = ["search(capital of France) -> 'Paris is the capital of France.'",
                "answer: Berlin, a historic city with a rich past"]

    absolute_scores = judge.evaluate_batch(task, [good_traj, bad_traj], mode="absolute")
    relative_scores = judge.evaluate_batch(task, [good_traj, bad_traj], mode="relative")
    print(f"  absolute mode (scored independently): {[round(s.score, 3) for s in absolute_scores]}")
    print(f"  relative mode (scored together, one call): {[round(s.score, 3) for s in relative_scores]}\n")

    # ── 4) Groundedness scoring (local judge, same rubric as judge_eval.py) ──
    print("── 4) Groundedness scoring (real local judge, GROUNDEDNESS_RUBRIC) ──")
    grounded_judge = LLMJudge(rollout_engine=rollout_engine, absolute_rubric=GROUNDEDNESS_RUBRIC)

    grounded_score = grounded_judge.evaluate_trajectory(task, good_traj)
    ungrounded_score = grounded_judge.evaluate_trajectory(task, bad_traj)
    print(f"  grounded answer (matches retrieved passage)  -> groundedness score = {grounded_score:.3f}")
    print(f"  ungrounded answer (contradicts retrieved passage) -> groundedness score = {ungrounded_score:.3f}")

In [5]:
if __name__ == "__main__":
    main()

── 1) Score clamping: necessity/frugality rewards ──


  searches= 1  necessity_reward=0.833  frugality_reward=0.250
  searches= 3  necessity_reward=0.500  frugality_reward=0.644
  searches=12  necessity_reward=0.000  frugality_reward=0.000
  -> reward does not grow (or drops) past the optimal search count: over-retrieval is capped, not free.

── 2) Prompt rotation: TrajectoryEvaluator's judge prompt ──


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

  rotation_id across 4 real judge calls: [1, 1, 1, 1]
  -> only one rotation id seen this run (still random each call), picked from 3 differently-worded judge prompts.

── 3) Relative scoring: LLMJudge mode='relative' vs 'absolute' ──


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


  absolute mode (scored independently): [0.8, 0.8]
  relative mode (scored together, one call): [0.95, 0.85]

── 4) Groundedness scoring (real local judge, GROUNDEDNESS_RUBRIC) ──


  grounded answer (matches retrieved passage)  -> groundedness score = 0.900
  ungrounded answer (contradicts retrieved passage) -> groundedness score = 0.000
